# Regenerate `supplementary_stats`

This notebook regenerates the manuscript-facing CSV summaries under `analysis_exports/supplementary_stats/` from the primary figure export folders and prediction outputs already present in the repository.

It writes regenerated files to `analysis_exports/supplementary_stats_regenerated/` and a validation report to `analysis_exports/supplementary_stats_regenerated/validation_report.csv`.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, t as student_t


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'analysis_exports').exists() and (candidate / 'Data').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from current working directory')

PROJECT_ROOT = find_project_root(Path.cwd())
OUT_DIR = PROJECT_ROOT / 'analysis_exports' / 'supplementary_stats_regenerated'
REF_DIR = PROJECT_ROOT / 'analysis_exports' / 'supplementary_stats'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, OUT_DIR, REF_DIR


In [ ]:
def save_csv(df: pd.DataFrame, name: str) -> Path:
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    return path


def compare_frames(name: str, generated: pd.DataFrame, reference: pd.DataFrame) -> dict:
    try:
        pd.testing.assert_frame_equal(
            generated.reset_index(drop=True),
            reference.reset_index(drop=True),
            check_dtype=False,
            atol=1e-10,
            rtol=1e-10,
        )
        return {'file': name, 'match': True, 'note': ''}
    except AssertionError as exc:
        return {'file': name, 'match': False, 'note': str(exc).splitlines()[0]}


def t_ci(series: pd.Series, alpha: float = 0.05) -> tuple[float, float]:
    series = pd.Series(series).dropna().astype(float)
    n = len(series)
    mean = float(series.mean())
    if n <= 1:
        return mean, mean
    sem = float(series.std(ddof=1) / np.sqrt(n))
    tcrit = float(student_t.ppf(1 - alpha / 2, df=n - 1))
    return mean - tcrit * sem, mean + tcrit * sem


In [ ]:
generated = {}

# Figure 2 dataset descriptives
labels = pd.read_pickle(PROJECT_ROOT / 'Data/Base/y_labels.pkl')
compound_ids = labels.index.to_series().str.replace(r'_[0-9]+$', '', regex=True)
rep_counts = compound_ids.value_counts().sort_index()
moa_count = int(labels.nunique())

generated['figure2_dataset_descriptives.csv'] = pd.DataFrame([
    {
        'n_samples': int(len(labels)),
        'n_compounds': int(compound_ids.nunique()),
        'n_moa_classes': moa_count,
        'replicates_min': int(rep_counts.min()),
        'replicates_median': float(rep_counts.median()),
        'replicates_max': int(rep_counts.max()),
        'pairwise_rows_no_aug': int(len(labels) * moa_count),
        'pairwise_rows_with_10_aug': int(len(labels) * moa_count * 10),
    }
])

# Figure 3c summaries
llm_dir = PROJECT_ROOT / 'analysis_exports/LLM_augmentation_experiment'
resample_metrics = pd.read_csv(llm_dir / 'resample_metrics.csv')
wide_pr = resample_metrics.pivot(index='resample_id', columns='method', values='pr_auc').dropna()
q_stat, p_val = friedmanchisquare(*[wide_pr[col].values for col in wide_pr.columns])
generated['figure3_panelc_friedman_check.csv'] = pd.DataFrame([
    {
        'metric': 'pr_auc',
        'friedman_Q': float(q_stat),
        'df': int(wide_pr.shape[1] - 1),
        'p': float(p_val),
        'n_cycles': int(wide_pr.shape[0]),
        'n_methods': int(wide_pr.shape[1]),
    }
])
generated['figure3_panelc_rm_anova_summary.csv'] = pd.read_csv(llm_dir / 'anova_summary.csv')[['metric', 'rm_anova_p']]
means = pd.read_csv(llm_dir / 'tukey_pr_auc_means.csv')[['method', 'pr_auc']].sort_values('pr_auc', ascending=False).reset_index(drop=True)
generated['figure3_panelc_pr_auc_method_means.csv'] = means

# Figure 3d summaries
text_dir = PROJECT_ROOT / 'analysis_exports/text_vs_notext_experiment'
tidy = pd.read_csv(text_dir / 'tidy_proteomics_cycles.csv')
paired = pd.read_csv(text_dir / 'paired_stats_per_model.csv')
wide = tidy.assign(group=tidy['Model'] + '|' + tidy['Method']).pivot_table(
    index=['cv_cycle', 'metric_key'], columns='group', values='value', aggfunc='mean'
).reset_index()
rows = []
for metric_key, metric_df in wide.groupby('metric_key'):
    vals = metric_df.drop(columns=['cv_cycle', 'metric_key'])
    q_stat, p_val = friedmanchisquare(*[vals[col].values for col in vals.columns])
    rows.append(
        {
            'metric_key': metric_key,
            'friedman_Q': float(q_stat),
            'df': int(vals.shape[1] - 1),
            'p': float(p_val),
            'n_cycles': int(len(metric_df)),
            'n_groups': int(vals.shape[1]),
        }
    )
generated['figure3_paneld_friedman_summary.csv'] = pd.DataFrame(rows).sort_values('metric_key').reset_index(drop=True)
generated['figure3_paneld_paired_ttests.csv'] = paired

# Figure 4 summaries
feat_root = PROJECT_ROOT / 'analysis_exports/Feature_augmentation_experiment'
generated['figure4_panela_tabm_friedman_pr_auc.csv'] = pd.read_csv(feat_root / 'tabm/pr_auc/tabm_friedman_pr_auc.csv')

TOP_FEATURES = [
    'CellPaint+ECFP+Proteomics',
    'CellPaint+ECFP+MIC+Proteomics',
    'CellPaint+ECFP+GC+MIC+Proteomics',
]

all_cycles = []
for model in ['rf', 'lightgbm', 'xgboost', 'mlp', 'pytorch_mlp', 'tabm']:
    df = pd.read_csv(feat_root / model / f'{model}_per_cycle_all_metrics.csv')
    df = df[df['var_tag'].isin(TOP_FEATURES)].copy()
    df['condition'] = model + '|' + df['var_tag'].astype(str)
    all_cycles.append(df[['cv_cycle', 'condition', 'pr_auc']])
all_cycles = pd.concat(all_cycles, ignore_index=True)

condition_means = (
    all_cycles.groupby('condition', as_index=False)['pr_auc']
    .mean()
    .rename(columns={'pr_auc': 'mean_pr_auc'})
    .sort_values('mean_pr_auc', ascending=False)
    .reset_index(drop=True)
)
generated['figure4_panelb_condition_means.csv'] = condition_means

top_conditions = condition_means['condition'].tolist()
wide_top = all_cycles[all_cycles['condition'].isin(top_conditions)].pivot_table(
    index='cv_cycle', columns='condition', values='pr_auc', aggfunc='mean'
)[top_conditions]
q_stat, p_val = friedmanchisquare(*[wide_top[col].values for col in wide_top.columns])
generated['figure4_panelb_global_friedman_summary.csv'] = pd.DataFrame([
    {
        'metric': 'pr_auc',
        'friedman_Q': float(q_stat),
        'df': int(wide_top.shape[1] - 1),
        'p': float(p_val),
        'n_cycles': int(len(wide_top)),
        'n_conditions': int(wide_top.shape[1]),
    }
])

winners = pd.read_csv(feat_root / 'summary_winners_all.csv')
generated['figure4_per_model_winners.csv'] = winners[
    ['model', 'metric', 'best', 'mean', 'std', 'n_cycles', 'friedman_p_raw', 'posthoc', 'mcs_size']
].rename(columns={'friedman_p_raw': 'friedman_p'})

# Figure 5 summaries
unc_root = PROJECT_ROOT / 'analysis_exports/uncertainty_estimator'
generated['figure5_eta_loo_overall_summary.csv'] = pd.read_csv(unc_root / 'eta_loo_overall_summary.csv')
generated['figure5_eta_loo_pairwise_holm.csv'] = pd.read_csv(unc_root / 'eta_loo_pairwise_stats_Holm.csv')

omni = {}
for line in (unc_root / 'eta_loo_omnibus_summary.txt').read_text().splitlines():
    if ': ' in line:
        k, v = line.split(': ', 1)
        omni[k] = v
generated['figure5_eta_loo_omnibus_summary.csv'] = pd.DataFrame([
    {
        'primary_metric': omni['primary_metric'],
        'methods_used': omni['methods_used'],
        'n_compounds': int(omni['n_compounds']),
        'omnibus_method': omni['omnibus_method'],
        'omnibus_stat': float(omni['omnibus_stat']),
        'omnibus_p': float(omni['omnibus_p']),
    }
])

fig5_ext = pd.read_csv(PROJECT_ROOT / 'predictions/Figure5_external_validation/Special_v1/uncertainty_meta_apply_semantic_allmod_fpr010/compound_uncertainty_meta.csv')
gen_fig5_ext = fig5_ext[['CompoundID', 'PredictedMoA', 'PredScore', 'Margin', 'p_error']].copy()
gen_fig5_ext['uncertain'] = fig5_ext['flag_meta'].astype(bool)
generated['figure5_external_compound_uncertainty.csv'] = gen_fig5_ext

# Figure 6 summaries
fig6_panels = {
    'a_eclipse_non_sig': PROJECT_ROOT / 'predictions/eclipse_data/non_sig_reduced/uncertainty_meta_apply/compound_uncertainty_meta.csv',
    'b_eclipse_sig': PROJECT_ROOT / 'predictions/eclipse_data/sig_reduced/uncertainty_meta_apply/compound_uncertainty_meta.csv',
    'c_external_non_sig': PROJECT_ROOT / 'predictions/external_data/non_sig_reduced/uncertainty_meta_apply/compound_uncertainty_meta.csv',
    'd_external_sig': PROJECT_ROOT / 'predictions/external_data/sig_reduced/uncertainty_meta_apply/compound_uncertainty_meta.csv',
}
fig6_frames = []
for panel, path in fig6_panels.items():
    df = pd.read_csv(path)
    sub = df[['CompoundID', 'PredictedMoA', 'PredScore', 'Margin', 'p_error']].copy()
    sub.insert(0, 'panel', panel)
    sub['uncertain'] = df['flag_meta'].astype(bool)
    fig6_frames.append(sub)
fig6_details = pd.concat(fig6_frames, ignore_index=True)
generated['figure6_panel_compound_details.csv'] = fig6_details
threshold = json.loads((PROJECT_ROOT / 'models/uncertainty_estimator_adapted/meta_artifact.json').read_text())['meta_threshold']
fig6_counts = fig6_details.groupby('panel', as_index=False).agg(
    n_compounds=('CompoundID', 'size'),
    uncertain_n=('uncertain', 'sum'),
)
fig6_counts['confident_n'] = fig6_counts['n_compounds'] - fig6_counts['uncertain_n']
fig6_counts['threshold'] = threshold
generated['figure6_panel_uncertainty_counts.csv'] = fig6_counts

# Supplementary Table 7 with 95% CIs
s7_rows = []
model_labels = {
    'pytorch_mlp': 'PyTorch-MLP',
    'mlp': 'MLP',
    'lightgbm': 'LightGBM',
    'tabm': 'TabM',
    'xgboost': 'XGBoost',
    'rf': 'RF',
}
for model in ['pytorch_mlp', 'mlp', 'lightgbm', 'tabm', 'xgboost', 'rf']:
    per_cycle = pd.read_csv(feat_root / model / f'{model}_per_cycle_all_metrics.csv')
    means = per_cycle.groupby('var_tag', as_index=False)['pr_auc'].mean().sort_values('pr_auc', ascending=False).head(3)
    for coalition in means['var_tag']:
        vals = per_cycle.loc[per_cycle['var_tag'] == coalition, 'pr_auc']
        ci_low, ci_high = t_ci(vals)
        s7_rows.append(
            {
                'Model': model_labels[model],
                'Coalition': coalition,
                'Mean PR AUC': float(vals.mean()),
                'SD': float(vals.std(ddof=1)),
                'n cycles': int(vals.shape[0]),
                '95% CI low': float(ci_low),
                '95% CI high': float(ci_high),
                '95% CI': f'[{ci_low:.4f}, {ci_high:.4f}]',
            }
        )
generated['table_s7_pr_auc_with_ci.csv'] = pd.DataFrame(s7_rows)

sorted(generated)


In [ ]:
# Write regenerated files
for name, df in generated.items():
    save_csv(df, name)

sorted(p.name for p in OUT_DIR.glob('*.csv'))


In [ ]:
# Compare regenerated files with the checked-in supplementary_stats folder
report_rows = []
for name, gen_df in generated.items():
    ref_path = REF_DIR / name
    if ref_path.exists():
        ref_df = pd.read_csv(ref_path)
        report_rows.append(compare_frames(name, gen_df, ref_df))
    else:
        report_rows.append({'file': name, 'match': False, 'note': 'reference file missing'})

validation_report = pd.DataFrame(report_rows).sort_values(['match', 'file'], ascending=[True, True]).reset_index(drop=True)
validation_report.to_csv(OUT_DIR / 'validation_report.csv', index=False)
validation_report


A `False` in the validation table means one of two things:

1. the checked-in `supplementary_stats` file is stale relative to the current primary exports or predictions, or
2. the manuscript-facing file was post-processed further before being copied into `supplementary_stats`.

The regenerated CSVs in `supplementary_stats_regenerated/` reflect the current repository state and document exactly how each summary was constructed from the checked-in sources.
